# Project End-to-End Documentation: Credit Risk Modeling & Portfolio Optimization
**Model Pipeline, Tracking Framework, and Financial Decision Optimization**

---

## 1. Executive Summary & Objective
In credit risk management, financial institutions must balance two competing forces: **Portfolio Growth** (maximizing loan approvals to drive interest revenue) and **Risk Mitigation** (restricting lending to minimize capital defaults). Setting an approval threshold too high chokes institutional revenue, while setting it too low invites catastrophic credit losses. 

This project implements an end-to-end machine learning pipeline to classify, predict, and financially optimize loan defaults. Using a comprehensive credit performance dataset sourced from Kaggle (originally originating from Coursera's "Loan Default Prediction Challenge"), we systematically engineer predictive features, train tree-based and linear architectures, rigorously calibrate probability outputs, and construct a graphical decision framework to maximize net portfolio returns.

---

## 2. Environment Setup, Initial Data Loading & Profiling
The pipeline begins by importing the core data science libraries (`pandas`, `numpy`, `seaborn`, `matplotlib`, `sklearn`) and establishing an isolated analytical workspace. 

### 2.1 Initial Data Loading
The raw historical loan performance records are ingested into a unified data structure. Initial code-driven inspections execute structural validations to verify row counts, column definitions, and protect against data corruption. This establishes a clean baseline dataframe for downstream manipulations.

### 2.2 Dataset Characteristics & Class Imbalance
A comprehensive profiling pass on the baseline dataset reveals the following critical parameters:
* **Missing Data:** The dataset contains **zero null values**, removing the need for statistical imputation strategies.
* **Feature Types:** The features consist of continuous financial metrics alongside exactly **4 categorical variables** that require numerical mapping.
* **Target Distribution:** The population demonstrates a baseline **loan default rate of 11%**. This signifies a substantial class imbalance (11% default vs. 89% non-default) which must be accounted for during training and evaluation via class-imbalance-resilient scoring metrics.

---

## 3. Feature Engineering & Categorical Transformations
To transform raw categorical strings into high-utility predictive assets, we bypass traditional one-hot encoding—which creates high dimensionality and masks feature interactions—and utilize advanced statistical transformations mapped strictly from the training dataset to prevent data leakage.

### 3.1 Hypothesis Testing: Individual Categorical Variables
We calculate the empirical probability of default for each individual category within our 4 categorical variables. We then conduct statistical hypothesis tests (such as Chi-Square Goodness-of-Fit) against the following criteria:
* **Null Hypothesis ($H_0$):** The default rate within a specific category equals the baseline population default rate ($11\%$).
* **Alternative Hypothesis ($H_1$):** The default rate within a specific category is significantly different from the baseline population default rate ($11\%$).

Confirming that specific categories deviate significantly from the 11% baseline proves that these variables hold explicit predictive power.

### 3.2 Hypothesis Testing: Stratified Categorical Variables
We extend this framework to **stratified (combined) categorical variables** by cross-tabulating and merging categorical traits to create unique borrower sub-segments (strata). Calculating empirical default rates for these combined segments reveals critical non-linear interaction patterns that individual variables hide in isolation.

### 3.3 Feature Selection via Information Value (IV)
To complement hypothesis testing and quantify predictive strength, we calculate the **Information Value (IV)** alongside **Weight of Evidence (WoE)** transformations using the standard credit risk formula:

$$IV = \sum \left( \% \text{ Non-Defaults} - \% \text{ Defaults} \right) \times \text{WoE}$$

* **Individual Baseline (Section 3.3):** Analysis shows that **`EmploymentType`** is the only single column demonstrating a meaningful signal, yielding an IV **greater than 0.2** (medium-to-strong predictive power). The remaining three standalone variables return weak or negligible predictive power on their own.
* **Stratified Analysis (Section 3.4):** Evaluating the cross-stratified categorical column yields a highly respectable, consolidated **Information Value of 0.14**. This shifts the engineered feature firmly into the medium predictive power tier, mathematically proving that feature combination extracts hidden interaction signals that individual columns miss.

### 3.4 Feature Creation: Loan-to-Income (LTI) Ratio
To provide the models with a clear proxy for debt burden, we programmatically engineer the **Loan-to-Income (LTI) Ratio** continuous feature:

$$\text{LTI Ratio} = \frac{\text{Loan Amount}}{\text{Annual Income}}$$

This standardizes financial leverage across diverse income tiers, handing the algorithm a direct splitting feature to isolate high-risk thresholds where debt outpaces earnings.

### 3.5 Reusable Pipeline: The `feature_engineering` Function
To ensure reproducible, production-ready execution, all transformations are encapsulated into a modular Python function: `feature_engineering(df, mapping_dict)`. It ingests raw data frames and applies pre-calculated WoE embeddings and probability mappings stored inside lightweight **Python dictionaries**. This lookup-table approach guarantees absolute feature parity across data splits while completely eliminating data leakage.

---

## 4. Analytical Model Selection: Architecture Comparison
The pipeline evaluates three distinct machine learning paradigms to handle the credit dataset:

### 4.1 Logistic Regression Pipeline
Because our upfront feature engineering creates highly informative stratified probability and linear WoE features, the categorical data shares an inherently linear relationship with the log-odds of the target default variable. This makes **Logistic Regression** an elegant, mathematically ideal model for this analysis. The architecture utilizes L2 (Ridge) regularization to control multicollinearity, and continuous fields are scaled to guarantee gradient descent convergence.

### 4.2 LightGBM (Gradient Boosting Tree Alternative)
As an alternative, we evaluate **LightGBM**, a highly optimized tree-like ensemble. LightGBM excels at uncovering high-order, non-linear interactions natively. It uses a leaf-wise growth strategy for rapid convergence and incorporates native balance weights (`is_unbalance=True`) to handle the 11% class imbalance directly.

### 4.3 CatBoost (Categorical Boosting Alternative)
We also evaluate **CatBoost**, an advanced symmetric decision tree framework. CatBoost handles raw categorical indices natively through an internal, randomized permutation technique for target encoding. This ordered boosting strategy actively fights prediction shift, ensuring robust stability and preventing overfitting on financial datasets.

---

## 5. Data Splitting & Experiment Tracking via MLflow
### 5.1 Data Partitioning Strategy
The dataset is split into three distinct subsets using a **stratified split** to ensure the 11% baseline default rate is precisely preserved across all partitions:
* **Training Set (70%):** Used exclusively to fit model parameters and calculate initial dictionary mappings.
* **Validation Set (15%):** Used for hyperparameter tuning and calibration fitting.
* **Test Set (15%):** Held back as a final "blind" dataset to evaluate real-world generalization.

### 5.2 MLflow Experimentation Setup
We run our training pipeline within a formalized experiment tracking framework using **MLflow** via a standardized `MLflowLogger` wrapper imported from our local `utils` module. The logging sequence automatically records class-imbalance-resilient performance metrics (**ROC-AUC, Precision, Recall, F1-Score**) across validation and test sets, saves vital model coefficients/feature importances, and stores visual diagnostic plots (Confusion Matrix, ROC Curve, and Precision-Recall Curve).

To audit the results, the user interface can be spun up from a notebook cell using the following bash command:
```bash
!python -m mlflow ui --host 127.0.0.1 --port 5000
```
The tracking dashboard is then instantly accessible via a web browser at **[http://127.0.0.1:5000](http://127.0.0.1:5000)**.

---

## 6. Model Calibration and Probability Validation
For credit risk scoring, ranking defaults is insufficient; models must output true **expected loss indicators**. If a model assigns a 30% default risk to a group of loans, exactly 30% of those applications should default in the real world. 

While tree-based models like CatBoost frequently distort raw probability scores (pushing them away from 0 and 1 due to iterative gradient boosting), linear models with upfront target encoding maintain high stability.

### 6.1 Calibration via CalibratedClassifierCV
To evaluate and correct these distortions, we implement post-processing probability scaling using Scikit-Learn’s `CalibratedClassifierCV(..., cv='prefit')`. It freezes the pre-trained base model weights and fits a probability correction function (Platt Scaling or Isotonic Regression) over the validation data.

### 6.2 Diagnostic Evaluation & The Brier Score
We visually evaluate probability alignment using **Calibration Plots (Reliability Diagrams)**, mapping predicted probabilities against true empirical default frequencies. To mathematically track calibration quality, we calculate the **Brier Score** (Mean Squared Error of probabilities):

$$\text{Brier Score} = \frac{1}{N} \sum_{i=1}^{N} (p_i - y_i)^2$$

### 7.3 Total Predicted Interest Optimization Function: Final Operational Verdict
Graphical analysis confirms that the Total Predicted Interest function reaches its peak at an approval threshold of **0.35**. 

Setting our credit-scoring engine's operational cut-off at exactly 0.35 achieves the optimal economic balance. It expands institutional market share safely while filtering out high-risk applicants before they cause significant capital losses, maximizing the net profitability of the credit portfolio.